#### [yt interrupt](https://youtu.be/nUW1FB_5vqo?list=PLVxiWMqQvhg9FCteL7I0aohj1_YiUx1x8&t=25)


#### include/entry.h
#### src/entry.S
//Exception vectors table 
- ventry label, is just to jump (branch) to the label
- ventry sync_invalid-el1t : will jump to sync_invalid_el1t:
  - correspond to entry.h

```
sync_invalid_el1t:
	handle_invalid_entry  SYNC_INVALID_EL1t
```

- handle_invalid_entry macro  
  - will call `kernel_entry`
  - pass 3 param  x0,x1,x2   
  - branch  to show_invalid_entry_message(c code:irq.c)
  - jump to err_hang

#### kernel exit  
- ventry	handle_el1_irq	
- handle_el1_irq:
	- `kernel_entry` 
	- bl	handle_irq  (c code:irq.c)
	- kernel_exit 

#### kernel_entry  
- 注意 sub sp , store sp 是減少  (stack pointer, stack  往下長)
- when we get exception, we jump out our regular context 
  - so we need to store the state of the processor (so we can jump back and restore prevois state)
  - stp : store all of our register to stack pointer (sp)
#### kernel exit  
- ldp back from stack  , add sp 是增加  (stack pointer)


### irq.h 
### include/irq.S
### include/peripherals/irq.h
BCM2835 docs  page 113
- irq table  
- 29 is aux interrupt

```c
enum vc_irqs {
    AUX_IRQ = (1 << 29)
};
```

### src/irq.c
所以基本上我們做的是 
在 irq.c 掛上一個 handle_irq  
假如 aux_interrupt 的時候就會接收 uart recv printf

- BCM2835 docs  page 15
- AUM_MU_IIR_REG register
- 10: receiver holds valid byte
- 注意  我們要把 uart_init  
 - REGS_AUX->mu_ier = 2; set to 2

### main.c will include "irq.h"
- enable_interrupt_controller();

### [yt interrupt timer](https://youtu.be/2dlBZoLCMSc?list=PLVxiWMqQvhg9FCteL7I0aohj1_YiUx1x8&t=87)
- [data sheet error](https://elinux.org/BCM2835_datasheet_errata#p12)
- so wer set miniuart ier to  from 2 to 0xD
  - 2:0010
  - D=13: 1101  

#### peripherals/irq.h  
#### irq.c  
```c
void enable_interrupt_controller() {
    REGS_IRQ->irq0_enable_1 = AUX_IRQ | SYS_TIMER_IRQ_1 | SYS_TIMER_IRQ_3;
}
and modify handle_ire()

掛上  handle_timer_1 , handle_timer_3 function
```
#### peripherals: timer.h
```c
#define CLOCKHZ 1000000
```
#### peripherals: timer.c
```c
timer_init()
- set timer interrupt
```

#### main.c  
initialzie the timer 